# BME 603/606: Computational Methods for Biological Modeling & Simulation

## In-Class Activity: Boolean Networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports done!")

---
## Part 1: Boolean Logic Gates

### Background

In a **Boolean network**, every node is a variable in {0, 1} (OFF / ON). Each node has an **update function** f_i:

    x_i(t+1) = f_i( x_1(t), ..., x_n(t) )

Two update schemes:
- **Synchronous:** all nodes update simultaneously each time step.
- **Asynchronous:** one node (chosen at random) updates per step — more realistic, since transcription and translation happen on different timescales.

The state space has 2^n states. Because the space is finite and the rules are deterministic, every trajectory eventually enters an **attractor** — either a **fixed point** (stable cell state) or a **limit cycle** (oscillation).

### Biological analogies

| Gate | Biological meaning | Example |
|---|---|---|
| **AND** | Co-activators (both required) | Gene = A AND B |
| **OR** | Redundant activators (one required) | Gene = A OR B |
| **NOT** | Repressor | Gene = NOT R |

In Python code, use `and`, `or`, `not` — not the `&&`, `||`, `!` shorthand.

### Exercise 1: Translate Biology into Boolean Rules

**Tasks:**
1. For each regulatory statement below, write the Boolean rule in Python. Use `and`, `or`, `not`.
   - **Co-activators:** G is on only if both A and B are on.
   - **Redundant activators:** G is on if either A or B is on.
   - **Repressor:** G is on only if R is off.
   - **Double negative:** G is on if R1 is off and R2 is off.
   - **Mixed:** G is on if A is on and R is off.
2. Evaluate each rule for all input combinations using `itertools.product` and print a truth table.
3. Verify the double-negative rule is logically equivalent to `not (r1 or r2)` — print both truth tables side by side.

> **Hint:** `itertools.product([0, 1], repeat=2)` gives all 2-input combinations: `(0,0), (0,1), (1,0), (1,1)`. Wrap Boolean results in `int()` before printing.

In [ ]:
# Your code here — define each rule as a lambda or function, then print truth tables

# Rule 1: co-activators (AND)
# rule_and = ...

# Rule 2: redundant activators (OR)
# rule_or = ...

# Rule 3: repressor (NOT)
# rule_not = ...

# Rule 4: double negative
# rule_double_neg = ...

# Rule 5: mixed (A AND NOT R)
# rule_mixed = ...


# Helper to print a truth table given a function and the input variable names
def print_truth_table(name, fn, inputs, n_inputs):
    """Print a truth table for a Boolean rule.

    Parameters
    ----------
    name    : str — label for the rule
    fn      : callable — takes a tuple of inputs and returns 0/1
    inputs  : list of str — variable names, in the same order as the tuple
    n_inputs: int — number of input variables
    """
    # Your code here — iterate product([0,1], repeat=n_inputs), print each row
    pass


# Your code here — call print_truth_table for each rule

# Your code here — verify double-negation equivalence


---
## Part 2: A Warm-Up 3-Node Network

### Background

From lecture, the 3-node network is:

    x_A(t+1) = x_B(t)
    x_B(t+1) = x_A(t) AND x_C(t)
    x_C(t+1) = NOT x_A(t)

The state-transition table from lecture shows **one fixed-point attractor** at (0,0,1) and **one 2-cycle** between (0,1,0) and (1,0,1).

**Background: Attractors and basins**

An **attractor** is a set of states that a trajectory can enter and never leave. Two types:

- **Fixed point:** x*(t+1) = x*(t) — a stable cell state (e.g., G1 arrest, apoptosis).
- **Limit cycle:** the network oscillates through a set of states of period p (e.g., cell cycle progression, circadian rhythms).

The **basin of attraction** of an attractor A is the set of all initial states that eventually flow to A:

    Basin(A) = { x(0) : lim_{t -> infinity} x(t) in A }

Large basins mean **robust cell fate** — many starting conditions lead to the same outcome.

### Exercise 2: Simulate Synchronously and Asynchronously

**Tasks:**
1. Complete `update_3node_sync(state)` returning the next state as a tuple of ints.
2. Simulate from `(0, 1, 0)` for 8 time steps synchronously. Print the trajectory. Do you see a 2-cycle?
3. Simulate from the same initial condition **asynchronously** — pick one node at random each step and update only that node. Run for 30 steps. Which attractor do you end up in?
4. Repeat step 3 with `np.random.seed(0)`, `np.random.seed(1)`, and `np.random.seed(2)`. Do different seeds reach different attractors?
5. Write `find_all_attractors(transitions)` that walks from every one of the 2^3 states and returns the set of attractors (as frozensets of tuples). Does your result match the lecture table?

> **Hint:** For the async version, only update `state[k]` for one random k in {0,1,2}; leave the other entries unchanged. For the attractor search, note that once you revisit a state you've seen in this trajectory, the cycle is the segment from the first visit onward.

In [ ]:
def update_3node_sync(state):
    """Synchronous update for the 3-node network from lecture.

    Parameters
    ----------
    state : tuple — (xA, xB, xC), each 0 or 1

    Returns
    -------
    tuple — next state (nA, nB, nC)
    """
    xA, xB, xC = state
    # Your code here — compute next values using the rules
    nA = None
    nB = None
    nC = None
    return (int(nA), int(nB), int(nC))


def update_3node_async(state, rng, node):
    """Asynchronous update: only `node` flips.

    Parameters
    ----------
    state : tuple — (xA, xB, xC)
    rng   : np.random.Generator or module — for reproducibility
    node  : int — which node (0, 1, or 2) to update

    Returns
    -------
    tuple — state with one entry replaced
    """
    # Your code here — compute the next value for `node` only
    pass


# Your code here — synchronous trajectory from (0,1,0), 8 steps

# Your code here — asynchronous trajectory, 30 steps, with three different seeds

# Your code here — find_all_attractors


### Exercise 3: A State-Transition Graph

The lecture showed a **state-transition graph (STG)** — a directed graph whose nodes are the 2^n states and whose edges are x -> f(x). It makes the attractors and their basins visually obvious.

**Tasks:**
1. Using `update_3node_sync`, build the STG as a dictionary `transitions = {state: next_state}` for all 8 states.
2. Print the transitions in a table (like the one in lecture) with columns: `A(t) B(t) C(t) -> A(t+1) B(t+1) C(t+1)`.
3. **Basin sizes:** for each attractor you found in Exercise 2, count how many starting states flow into it. Verify the basins partition the 8-state space.
4. **Async vs. sync basins.** In a few sentences, discuss: does the async trajectory always reach the same attractor as the sync one from the same initial condition? Why or why not?

> **Hint:** To find which attractor a state flows into, follow transitions until you revisit a state, then identify the cycle or fixed point from the visited-history.

In [ ]:
# Your code here — build STG, print the transitions table

# Your code here — compute basin sizes for each attractor


**Your comparison of async vs. sync attractors:**

*Write your answer here — did the async trajectory reach the same attractor as sync in each seed? Do you expect them to always agree?*

---
## Part 3: The MAPK/ERK Cascade

### Background

The MAPK cascade transmits growth-factor signals from the cell surface via **EGFR -> Ras -> Raf -> MEK -> ERK**. ERK feeds back to inhibit Raf, creating a **negative-feedback loop** that can generate oscillations.

The rules (with `GF` as a fixed external input):

    EGFR(t+1) = GF
    Ras(t+1)  = EGFR
    Raf(t+1)  = Ras AND NOT ERK
    MEK(t+1)  = Raf
    ERK(t+1)  = MEK

Note that `GF` is **not** a state variable — it is a constant input.

### Exercise 4: Simulate the MAPK Cascade

**Tasks:**
1. Implement `update_mapk(state, GF=1, ERK_forced=None)` returning a 5-tuple `(EGFR, Ras, Raf, MEK, ERK)`. `ERK_forced` (when not `None`) overrides ERK's next value — use it for knock-out experiments.
2. Simulate synchronously from `(0, 0, 0, 0, 0)` for 20 steps with `GF = 1`. Plot all five species vs. time step (use `plt.step(..., where='post')` for a clean digital look).
3. Do you see the oscillation in ERK shown in lecture? What is the **period** of the limit cycle?
4. Repeat with `GF = 0`. What is the attractor? Explain in terms of the cascade.
5. **Perturbation — ERK knock-out.** Simulate with `ERK_forced=0` and `GF=1`. What does the Raf -> MEK -> ERK cascade do now? Why does removing the feedback loop change the dynamics so dramatically?

> **Hint:** When interpreting the period, count the number of time steps between two consecutive rising edges of ERK.

In [ ]:
def update_mapk(state, GF=1, ERK_forced=None):
    """Update the 5-node MAPK cascade.

    Parameters
    ----------
    state      : tuple — (EGFR, Ras, Raf, MEK, ERK)
    GF         : int — growth factor (external input, 0 or 1)
    ERK_forced : int or None — if not None, force ERK's next value

    Returns
    -------
    tuple — next state (EGFR, Ras, Raf, MEK, ERK)
    """
    EGFR, Ras, Raf, MEK, ERK = state
    # Your code here — compute the five next values
    nEGFR = None
    nRas = None
    nRaf = None
    nMEK = None
    nERK = None
    if ERK_forced is not None:
        nERK = ERK_forced
    return (int(nEGFR), int(nRas), int(nRaf), int(nMEK), int(nERK))


def simulate_mapk(init, n_steps=20, GF=1, ERK_forced=None):
    """Return an array of shape (n_steps+1, 5)."""
    # Your code here — iterate update_mapk, collect states
    pass


def plot_mapk(traj, title="MAPK/ERK cascade"):
    """Stepped plot of all 5 species vs. time step."""
    # Your code here
    pass


# Your code here — simulate GF=1 and plot

# Your code here — simulate GF=0 and plot

# Your code here — ERK knock-out (ERK_forced=0)


**Your interpretation of the ERK knock-out:**

*Write your answer here — what changed in the dynamics, and why?*

---
## Part 4: The Mammalian Cell Cycle (10-node network)

### Background

Cell-cycle progression is driven by **cyclin-CDK complexes** and checkpoint proteins. The Boolean model below (an extension of Fauré et al., 2006) includes the APC/C regulators **Cdc20, Cdh1, and UbcH10** in addition to the core cyclins and Rb/E2F/p27 machinery.

**Nodes (in order):** `CycD, Rb, E2F, CycE, CycA, p27, Cdc20, Cdh1, UbcH10, CycB`

**Update rules** (Python `and`, `or`, `not`):

    CycD   = CycD
    Rb     = (not CycA and not CycB and not CycD and not CycE) or (p27 and not CycB and not CycD)
    E2F    = (not Rb and not CycA and not CycB) or (p27 and not Rb and not CycB)
    CycE   = E2F and not Rb
    CycA   = (E2F and not Rb and not Cdc20 and not (Cdh1 and UbcH10)) or (CycA and not Rb and not Cdc20 and not (Cdh1 and UbcH10))
    p27    = (not CycD and not CycE and not CycA and not CycB) or (p27 and not (CycE and CycA) and not CycB and not CycD)
    Cdc20  = CycB
    Cdh1   = (not CycA and not CycB) or (Cdc20) or (p27 and not CycB)
    UbcH10 = not Cdh1 or (Cdh1 and UbcH10 and (Cdc20 or CycA or CycB))
    CycB   = not Cdc20 and not Cdh1

**`CycD`** is an **external input** (mitogenic signal). Fix it at 1 (ON) for the standard simulation.

The self-reference terms (`CycD = CycD`, `CycA = ... or (CycA and ...)`, `p27 = ... or (p27 and ...)`, `UbcH10 = ... or (Cdh1 and UbcH10 and ...)`) encode **positive autoregulation / memory**. Include them exactly as written — they materially affect dynamics.

### Exercise 5: Implement and Simulate the Cell-Cycle Network

**Tasks:**
1. Implement `update_cell_cycle(state, CycD=1, fixed=None)` returning a 10-tuple in the node order above. The `fixed` parameter is an optional dict mapping node names to values (e.g., `{"Rb": 0}`) for perturbations.
2. Start from the **quiescent state** `(1, 1, 0, 0, 0, 1, 0, 1, 0, 0)` — CycD and Rb high, p27 and Cdh1 high, everything else low. Simulate synchronously for 30 steps. Plot all 10 nodes as a stepped time series.
3. Do you see periodic cycling? Estimate the **period** of the attractor. Compare the trajectory shape to the one shown in lecture.
4. **Identify the phases.** For the trajectory you generated, annotate which time steps correspond to G1, S, and G2/M based on which cyclins are high:
   - G1: CycD high, Rb high
   - S: CycE and/or CycA high
   - G2/M: CycB high
5. **Run asynchronously.** Update one random internal node per step, run for 500 steps, and record the sequence of states. Do **not** let the random update flip `CycD` — it is an input. Plot the 10-node time series. Is the cycle still recognizable? Is the period the same, longer, or stochastic?
6. **Attractor and basin enumeration.** Fix `CycD = 1`. Enumerate all 2^9 = 512 states of the 9 internal nodes, follow each synchronously, and classify the terminal state:
   - Which terminal states are **fixed points**?
   - Which are **limit cycles**? Record the period of each.
   - Report the **basin size** of each attractor.
7. **Interpret.** Which attractor has the biggest basin? Does that match the biological expectation that the G1/quiescent state is the most robust cell fate?

> **Hint:** For `update_cell_cycle`, unpack the state into a dict keyed by node name; compute each `next_<node>` using the rules; apply `fixed` overrides last; return values in the original node order. `CycD` in the dict always comes from the `CycD` argument (so a perturbation `{"CycD": 0}` is equivalent to calling with `CycD=0`).

In [ ]:
NODES = ['CycD', 'Rb', 'E2F', 'CycE', 'CycA', 'p27', 'Cdc20', 'Cdh1', 'UbcH10', 'CycB']


def update_cell_cycle(state, CycD=1, fixed=None):
    """Synchronous update for the 10-node cell-cycle network.

    Parameters
    ----------
    state : tuple of 10 ints — in NODES order
    CycD  : int — external mitogenic input (0 or 1)
    fixed : dict or None — node name -> forced value (applied after update)

    Returns
    -------
    tuple of 10 ints — next state in NODES order
    """
    d = dict(zip(NODES, state))
    d['CycD'] = CycD  # input override
    # Your code here — compute each next value using the rules
    nxt = {}
    # nxt['CycD'] = ...
    # nxt['Rb'] = ...
    # nxt['E2F'] = ...
    # nxt['CycE'] = ...
    # nxt['CycA'] = ...
    # nxt['p27'] = ...
    # nxt['Cdc20'] = ...
    # nxt['Cdh1'] = ...
    # nxt['UbcH10'] = ...
    # nxt['CycB'] = ...

    # Apply fixed overrides
    if fixed:
        for k, v in fixed.items():
            nxt[k] = v
        nxt['CycD'] = fixed.get('CycD', CycD)

    return tuple(int(nxt[n]) for n in NODES)


def simulate_cell_cycle(init, n_steps=30, CycD=1, fixed=None):
    """Return an array of shape (n_steps+1, 10)."""
    # Your code here
    pass


def plot_cell_cycle(traj, title="Cell cycle"):
    """Stepped plot of all 10 nodes."""
    # Your code here — one subplot or stacked lines, with NODES as labels
    pass


QUIESCENT = (1, 1, 0, 0, 0, 1, 0, 1, 0, 0)

# Your code here — synchronous simulation and plot

# Your code here — asynchronous simulation and plot

# Your code here — attractor and basin enumeration (CycD=1)


**Your attractor and basin summary:**

*List each attractor (fixed point or cycle with period), its basin size, and which cell-cycle phase it corresponds to. Which attractor dominates? Does that match the biological expectation?*

---
## Part 5: Perturbation Analysis

### Background

In a Boolean network, a **perturbation** means **fixing a node** to a constant value and re-running the dynamics:
- **Knock-out:** fix the node to 0.
- **Overexpression:** fix the node to 1.

From lecture:

| Perturbation | Effect on rules | Predicted phenotype |
|---|---|---|
| Rb knock-out | `Rb = 0` | E2F constitutively active -> uncontrolled S-phase entry |
| CycD = 0 | `CycD = 0` | All states -> G0/G1 arrest |
| CycB overexpression | `CycB = 1` | Mitosis locked, fails to exit |
| p27 knock-out | `p27 = 0` | CycE/CycA inhibition removed -> accelerated cycling |

### Exercise 6: Knockouts and Overexpression

**Tasks:**
1. Write a helper `run_perturbation(fixed, n_steps=30, init=QUIESCENT)` that simulates from `init` under the perturbation and returns the trajectory.
2. Run the four perturbations from lecture:
   - **Rb knock-out** (`{"Rb": 0}`)
   - **CycD = 0** (`{"CycD": 0}`)
   - **CycB overexpression** (`{"CycB": 1}`)
   - **p27 knock-out** (`{"p27": 0}`)
3. For each perturbation, plot the trajectory and label the resulting attractor (fixed point vs. cycle, period, which phase).
4. **Compare** your simulation results to the predicted phenotypes from lecture. Do all four predictions hold? Which ones differ, and why?
5. **Design your own perturbation.** Pick one more node (`Cdh1`, `UbcH10`, `E2F`, or `Cdc20`) and predict what will happen **before** you run the simulation. Then run it and compare. Write your prediction and outcome in the markdown cell below.

> **Hint:** Some perturbations can send the network to a **fixed point** (arrest) or a **trivial cycle** of period 1-2. Classify carefully — a period-2 oscillation is a limit cycle, not a fixed point.

In [ ]:
def run_perturbation(fixed, n_steps=30, init=QUIESCENT):
    """Simulate the cell cycle under a perturbation.

    Parameters
    ----------
    fixed   : dict — node name -> forced value
    n_steps : int — number of time steps
    init    : tuple — starting state

    Returns
    -------
    trajectory : np.ndarray of shape (n_steps+1, 10)
    """
    # Your code here — use simulate_cell_cycle with fixed=fixed
    pass


PERTURBATIONS = {
    "Rb knock-out":        {"Rb": 0},
    "CycD = 0":            {"CycD": 0},
    "CycB overexpression": {"CycB": 1},
    "p27 knock-out":       {"p27": 0},
}

# Your code here — run each perturbation, plot, and report the attractor

# Your code here — your own custom perturbation


**Your prediction for the custom perturbation:**

*Write your answer here — which node did you pick, what did you expect, and what actually happened?*

---
## Part 6: Compare Against BooleSim

The [BooleSim](https://matthiasbock.github.io/BooleSim/) web tool has a **cell-cycle demo** that implements a Boolean model of the cell cycle. Let's cross-check our implementation against it.

### Exercise 7: Reconcile Our Model with BooleSim

**Tasks:**
1. Open https://matthiasbock.github.io/BooleSim/ in a browser.
2. Click **Import** and load the **cell cycle demo**.
3. Identify which nodes are present in BooleSim's demo. Do they match our 10-node list (`CycD, Rb, E2F, CycE, CycA, p27, Cdc20, Cdh1, UbcH10, CycB`)?
4. Run BooleSim with its default input state and record the trajectory. Then set the initial state in your notebook to match BooleSim's and re-run your synchronous simulation.
5. Compare node by node. Where do the trajectories agree? Where do they diverge?
6. **Diagnose any discrepancies.** Common causes:
   - A node treated as a **constant input** in one model but as an **internal node** in the other (e.g., `CycD`, `Cdc20`).
   - A difference in rule structure — e.g., `p27 = ... or (p27 and ...)` gives p27 self-memory (positive autoregulation); if BooleSim drops the self-term, dynamics differ.
   - **Update semantics** — BooleSim may default to a specific schedule. Try both synchronous and asynchronous if the demo offers it.
7. Once you've reconciled the two, **document the differences** in the markdown cell below.

> **Hint:** BooleSim's demo may be a **subset** of our network (e.g., 7 or 8 nodes rather than 10). It's fine if the networks differ — the exercise is to identify **exactly how**, not to make them identical. Also: BooleSim uses a specific default update function; check its documentation or the tool's on-screen info.

In [ ]:
# Your code here — set the initial state to match BooleSim's demo,
# re-run our simulation, and compare trajectories side by side

# Your code here — if BooleSim's demo uses a subset of nodes,
# optionally implement that subset for a direct comparison


**Your comparison of our network vs. BooleSim's demo:**

*Write your answer here — what nodes matched, what differed (which were inputs vs. internal nodes, which self-loops were present, what update semantics BooleSim uses), and whether trajectories agree where the networks overlap.*

---
## Part 7: Challenge Exercise (Bonus)

### Exercise 8: Build a GINsim Model from Scratch

**For bonus credit.** The GINsim model repository at https://ginsim.github.io/models/ hosts dozens of published Boolean models. Pick **one model** whose logic is fully documented, then **re-implement and simulate it in this notebook**.

**Selection guidance:**
- Prefer a model with a **diagram (regulatory graph)**, a **PDF paper**, or a **text description** of the rules — the `.zginml` file is XML and readable only with a tool.
- Reasonable sizes: **5-15 nodes**.
- Classic choices: the *Drosophila* segment-polarity network (von Dassow, 2000), the Th-lymphocyte differentiation network, or the apoptosis network.

**Tasks:**
1. Pick a model. In a markdown cell, record: model name, citation, number of nodes, and a link to the regulatory graph or paper.
2. **Transcribe the rules.** For each node, write its update function as a Python expression using `and`, `or`, `not`. Include self-loops (self-activation or self-repression) exactly as written — they materially change dynamics.
3. **Implement the update function** `update_<model>(state, **inputs)` with any external inputs exposed as keyword arguments.
4. **Simulate synchronously** from at least two distinct initial conditions (e.g., a "resting" and a "stimulated" state from the paper). Plot the trajectories.
5. **Enumerate attractors.** For all 2^n states, classify the terminal state. Report:
   - How many fixed points and how many limit cycles,
   - The period of each cycle,
   - The basin size of each attractor.
6. **Perturbation.** Pick one node and knock it out. Does the attractor landscape change? Which attractors disappear, which appear, and which basins shrink or grow?
7. **Sanity-check against the paper.** The original paper likely reports expected attractors (e.g., "two stable states corresponding to Th1 and Th2 fates"). Do your results match? If not, diagnose.

**Deliverable:** a working notebook cell that simulates the model, an attractor table, a perturbation plot, and a short discussion comparing results to the published paper.

> **Hints:**
> - Parse the `.zginml` XML only if you must — transcribe rules manually from the published description.
> - Use `frozenset` for attractors so fixed points and cycles can be compared as sets.
> - If the model has external inputs, fix them at 1 for the "stimulated" condition and 0 for "resting", treating them as constants (not state variables) in the 2^n enumeration.
> - For attractor enumeration, memoize trajectories from each start state — you only need to follow **new** states.

In [ ]:
# Your code here — build the model, simulate, enumerate attractors, perturb

MODEL_NODES = []  # fill in


def update_model(state, **inputs):
    """Synchronous update for the chosen GINsim model.

    Parameters
    ----------
    state  : tuple — current values of the model nodes
    inputs : dict — external inputs, keyed by name

    Returns
    -------
    tuple — next state
    """
    # Your code here
    pass


def enumerate_attractors(update_fn, n_nodes, **inputs):
    """Walk every state and return dict {attractor_frozenset: [basin_states]}.

    Parameters
    ----------
    update_fn : callable — state tuple -> next state tuple
    n_nodes   : int — number of nodes
    inputs    : dict — passed through to update_fn

    Returns
    -------
    dict — frozenset of state tuples -> list of basin state tuples
    """
    # Your code here
    pass


# Your code here — run everything


**Your discussion of the GINsim model:**

*Write your answer here — model name, citation, node count, attractors found, how your results compare to the published paper, and what the perturbation experiment revealed.*

---
## Further Exploration

### BooleSim

[BooleSim](https://matthiasbock.github.io/BooleSim/) is a lightweight browser tool for building small Boolean networks by hand. It supports synchronous and asynchronous updates and visualizes state-transition graphs.

### GINsim

[GINsim](https://ginsim.github.io/) is a full-featured Boolean network editor and simulator with export to SBML-qual. It ships with a large curated model repository and supports structural analysis (feedback loops, stable states, trap spaces).

### PyBoolNet

[PyBoolNet](https://github.com/hklarner/PyBoolNet) is the Python library for Boolean network analysis — attractor detection, trap space enumeration, model checking. Natural next step if you continue with Boolean networks.

### Related Methods

- **Continuous limit.** As the Hill coefficient n -> infinity, the activation function f_act([TF]) approaches a step function — a Boolean network is the **infinite-cooperativity limit** of an ODE model.
- **Probabilistic Boolean networks (PBNs).** Allow each node to choose stochastically among multiple update functions, useful when the true regulatory logic is uncertain.
- **SBML-qual.** The standard exchange format for qualitative (Boolean) models.

### References

- [Boolean network — Wikipedia](https://en.wikipedia.org/wiki/Boolean_network)
- [BooleSim — Bock et al.](https://matthiasbock.github.io/BooleSim/)
- [GINsim model repository](https://ginsim.github.io/models/)
- [PyBoolNet documentation](https://pyboolnet.readthedocs.io/)
- Kauffman, S. A. (1969). Metabolic stability and epigenesis in randomly constructed genetic nets. *J. Theor. Biol.*, 22(3), 437-467.
- Fauré, A., Naldi, A., Chaouiya, C., & Thieffry, D. (2006). Dynamical analysis of a generic Boolean model for the control of the mammalian cell cycle. *Bioinformatics*, 22(14), e124-e131.
- Elowitz, M. B. & Leibler, S. (2000). A synthetic oscillatory network of transcriptional regulators. *Nature*, 403, 335-338.